# P3(수정). 법 제12조 블록 — 완전한 문장으로 교체 + 신고 주체 결정 기록

## 결정 기록 (2026-11-04, 사용자 승인)
| 질문 | 결정 | 근거 |
|---|---|---|
| Q1 학교 담당교사의 보건소 신고 | **지침의 학교 행동**으로 분류. 법적 의무 여부는 제12조 조건 확인 전까지 단정하지 않음 | 홍역 지침 93쪽 |
| Q2 주체 표기 | **"학교[담당교사]"** (지침 표기). 법 제12조의 "관리인·경영자 또는 대표자"는 별도 주체로 두고 같은 사람으로 추정하지 않음 | 지침 93쪽, 법 제12조 ① 제2호 |
| Q3 의료기관 신고 | `SchoolAction` 추출에서 **제외**. 학교 행동만 추출 | 개발계획 추출 규칙, 법 제11조 |
| Q4 "보건복지부령으로 정하는 감염병"에 홍역 포함 여부 | **확인 전**. 2차 자료(시행규칙 제6조 목록에 홍역 표시)는 2016~2019년 개정 자료라 현행 확인이 안 됨. 국가법령정보센터의 시행규칙 최신본을 확인해야 함 | 웹 검색 결과 |

## 블록 교체
- 기존 `p0019:b01`(제12조 ① 앞부분), `p0019:b02`(제12조 ① 제2호 일부)는 잘려 있었다.
- 19쪽 원문으로 `p0019:b01` = 제12조 ①(전체), `p0019:b02` = 제12조 ②로 다시 만든다.
- `p0019:b03`(제13조 ①)은 그대로 둔다.

In [1]:
import json, os
from preprocess.prevention_legal.pages import DOCS, excerpt_in_page

KEY = "law"
DOC_ID = DOCS[KEY][0]
BLOCKS = "../../data/interim/prevention_legal/blocks.jsonl"
TARGETS = [f"{DOC_ID}:p0019:b01", f"{DOC_ID}:p0019:b02"]

NEW = {
    TARGETS[0]: ("제12조(그 밖의 신고의무자) ① 다음 각 호의 어느 하나에 해당하는 사람은 제1급감염병부터 제3급감염병까지에 해당하는 "
                 "감염병 중 보건복지부령으로 정하는 감염병이 발생한 경우에는 의사, 치과의사 또는 한의사의 진단이나 검안을 요구하거나 "
                 "해당 주소지를 관할하는 보건소장에게 신고하여야 한다. "
                 "1. 일반가정에서는 세대를 같이하는 세대주. 다만, 세대주가 부재 중인 경우에는 그 세대원 "
                 "2. 학교, 사회복지시설, 병원, 관공서, 회사, 공연장, 예배장소, 선박ㆍ항공기ㆍ열차 등 운송수단, 각종 사무소ㆍ사업소, "
                 "음식점, 숙박업소 또는 그 밖에 여러 사람이 모이는 장소로서 보건복지부령으로 정하는 장소의 관리인, 경영자 또는 대표자 "
                 "3. 「약사법」에 따른 약사ㆍ한약사 및 약국개설자"),
    TARGETS[1]: ("② 제1항에 따른 신고의무자가 아니더라도 감염병환자등 또는 감염병으로 인한 사망자로 의심되는 사람을 발견하면 보건소장에게 알려야 한다."),
}
SECTION = {
    TARGETS[0]: "제3장 신고 및 보고 · 제12조(그 밖의 신고의무자) ①",
    TARGETS[1]: "제3장 신고 및 보고 · 제12조(그 밖의 신고의무자) ②",
}
NOTE = "감염병예방법 2026.10.8 시행본 19쪽 · 보건복지부령 감염병 해당 여부: 확인 전 (시행규칙 현행본 미확인)"
PIECES = {
    TARGETS[0]: ["다음 각 호의 어느 하나에 해당하는 사람은", "보건복지부령으로 정하는 감염병이 발생한 경우에는",
                 "해당 주소지를 관할하는 보건소장에게 신고하여야 한다", "학교, 사회복지시설, 병원, 관공서", "관리인, 경영자 또는 대표자"],
    TARGETS[1]: ["제1항에 따른 신고의무자가 아니더라도", "보건소장에게 알려야 한다"],
}

In [2]:
# 1) 원문 확인 (19쪽, 조각별). 하나라도 없으면 저장하지 않는다.
for cid in TARGETS:
    for piece in PIECES[cid]:
        ok = excerpt_in_page(piece, KEY, 19)
        print(("원문 확인 " if ok else "확인 필요 "), cid[-3:], "|", piece[:30])
        assert ok, f"19쪽에서 조각을 찾지 못했습니다: {piece}"

원문 확인  b01 | 다음 각 호의 어느 하나에 해당하는 사람은
원문 확인  b01 | 보건복지부령으로 정하는 감염병이 발생한 경우에는
원문 확인  b01 | 해당 주소지를 관할하는 보건소장에게 신고하여야 한다
원문 확인  b01 | 학교, 사회복지시설, 병원, 관공서
원문 확인  b01 | 관리인, 경영자 또는 대표자
원문 확인  b02 | 제1항에 따른 신고의무자가 아니더라도
원문 확인  b02 | 보건소장에게 알려야 한다


In [3]:
# 2) 교체: 두 블록만 바꾸고 나머지는 그대로 둔다 (임시 파일 → 원자적 교체)
with open(BLOCKS, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]
before = {r["chunk_id"]: r for r in records}
assert all(cid in before for cid in TARGETS), "교체 대상이 없습니다."

out = []
for r in records:
    cid = r["chunk_id"]
    if cid in NEW:
        out.append({"chunk_id": cid, "document_id": DOC_ID, "text": NEW[cid],
                    "pdf_page": 19, "printed_page": None, "section": SECTION[cid], "source_note": NOTE})
    else:
        out.append(r)
ids = [r["chunk_id"] for r in out]
assert len(ids) == len(set(ids)), "chunk_id 중복"

tmp = BLOCKS + ".tmp"
with open(tmp, "w", encoding="utf-8") as f:
    for r in out:
        f.write(json.dumps(r, ensure_ascii=False) + chr(10))
os.replace(tmp, BLOCKS)

after = {json.loads(l)["chunk_id"]: json.loads(l) for l in open(BLOCKS, encoding="utf-8") if l.strip()}
changed = sorted(c for c in before if before[c] != after.get(c))
print("전체 블록:", len(after), "| 바뀐 블록:", changed)
assert changed == sorted(TARGETS)
print("나머지", len(after) - len(TARGETS), "개는 그대로 ✅")

전체 블록: 25 | 바뀐 블록: ['moleg_infectious_disease_act_21521:p0019:b01', 'moleg_infectious_disease_act_21521:p0019:b02']
나머지 23 개는 그대로 ✅
